## Particle Swarm Animation

In [2]:
import random
import pygame
import sys
import numpy as np

# PSO parameters
MAX_VELOCITY = 1.0  # Maximum velocity for particles
BOUNDS = [-5, 5]    # Search space bounds for x and y


In [ ]:
import random
import pygame
import sys
import numpy as np

class Particle:
    def __init__(self, position):
        self.position =   position
        self.velocity =   [random.uniform(-MAX_VELOCITY, MAX_VELOCITY) for _ in range(2)]  # Initial random velocity
        self.p_best =     position  # Best position found by this particle
        self.p_best_fitness =   self.fitness()  # Fitness of the best position

    def fitness(self):
        # Calculate the value of the function f(x, y) = x^2 + y^2
        return self.position[0]**2 + self.position[1]**2


    def update(self, g_best, c1=1, c2=1):
        for i in range(2):
            # Update velocity
            self.velocity[i] += c1 * random.random() * (self.p_best[i] - self.position[i]) + c2 * random.random() * (g_best[i] - self.position[i])
            
            # Limit velocity to avoid explosion using numpy.clip
            self.velocity[i] = np.clip(self.velocity[i], -MAX_VELOCITY, MAX_VELOCITY)

            # Update position
            self.position[i] += self.velocity[i]

            # Constrain position to search space bounds using numpy.clip
            self.position[i] = np.clip(self.position[i], BOUNDS[0], BOUNDS[1])


        # Update personal best if the current position is better
        current_fitness = self.fitness()
        if current_fitness < self.p_best_fitness:
            self.p_best = self.position.copy()
            self.p_best_fitness = current_fitness


In [12]:
class Swarm:
    def __init__(self, num_particles):
        # Initialize particles with random positions
        self.swarm = []
        for _ in range(num_particles):
            position = [random.uniform(BOUNDS[0], BOUNDS[1]), random.uniform(BOUNDS[0], BOUNDS[1])]
            self.swarm.append(Particle(position))


    def find_global_best(self):
        # Find the particle with the best fitness
        return min((p.p_best for p in self.swarm), key=lambda x: sum(xi ** 2 for xi in x))

    def fitness(self, position):
        # Calculate the value of the function f(x, y) = x^2 + y^2
        return sum((position[i] ** 2 for i in range(2)))

    def update(self):
        for particle in self.swarm:
            particle.update(self.g_best)


    def simulate(self):
        """Generator function to yield the swarm state at each step."""
        while True:
            self.g_best = self.find_global_best()
            self.update()
            yield self.swarm, self.g_best


In [16]:
class Animation:
    def __init__(self, swarm, width=800, height=600):
        self.swarm = swarm
        self.width = width
        self.height = height

        # Initialize Pygame
        pygame.init()
        self.screen = pygame.display.set_mode((width, height))
        pygame.display.set_caption("Particle Swarm Optimization")

        # Colors
        self.WHITE = (255, 255, 255)
        self.BLACK = (0, 0, 0)
        self.RED = (255, 0, 0)

        # Scaling factors (to map function space to screen space)
        self.SCALE_X = width / 10  # Maps x from [-5, 5] to [0, width]
        self.SCALE_Y = height / 10  # Maps y from [-5, 5] to [0, height]
        self.OFFSET_X = width // 2  # Centers x = 0
        self.OFFSET_Y = height // 2  # Centers y = 0

    def draw_particle(self, position):
        # Map particle position to screen coordinates:
        x = int(self.OFFSET_X + position[0] * self.SCALE_X)
        y = int(self.OFFSET_Y - position[1] * self.SCALE_Y)
        # Negative y to flip the axis: TODO
        pygame.draw.circle(self.screen, self.RED, (x, y), 5)

    def draw_global_best(self, position):
        # Map global best position to screen coordinates
        x = int(self.OFFSET_X + position[0] * self.SCALE_X)
        y = int(self.OFFSET_Y - position[1] * self.SCALE_Y)
        pygame.draw.circle(self.screen, self.BLACK, (x, y), 7)
      

    def run(self):
        clock = pygame.time.Clock()
        swarm_generator = self.swarm.simulate()  # Get the generator

        running = True
        while running:
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    running = False

            # Get the next state of the swarm
            particles, g_best = next(swarm_generator)
            clock.tick(10)  # Limit to 10 frames per second

            # Draw everything
            self.screen.fill(self.WHITE)
            for particle in particles:
                self.draw_particle(particle.position)
            self.draw_global_best(g_best)
            pygame.display.flip()

        pygame.quit()
        sys.exit()

In [17]:
swarm = Swarm(20)  # Create a swarm with 20 particles
animation = Animation(swarm)  # Create the animation
animation.run()  # Run the animation

SystemExit: 